# 05 — Text Quality & Relevance

32 examples covering ReadingLevel, ReadingTime, GibberishText, RedundantSentences,
SaliencyCheck, ExtractedSummarySentencesMatch, SimilarToDocument, RelevancyEvaluator.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv
guardrails hub install hub://guardrails/reading_level
guardrails hub install hub://guardrails/reading_time
guardrails hub install hub://guardrails/gibberish_text
guardrails hub install hub://guardrails/redundant_sentences
guardrails hub install hub://guardrails/saliency_check
guardrails hub install hub://guardrails/extracted_summary_sentences_match
guardrails hub install hub://guardrails/similar_to_document
guardrails hub install hub://guardrails/relevancy_evaluator
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
# Install Hub validators (run once)
!guardrails hub install hub://guardrails/reading_level --quiet
!guardrails hub install hub://guardrails/reading_time --quiet
!guardrails hub install hub://guardrails/gibberish_text --quiet
!guardrails hub install hub://guardrails/redundant_sentences --quiet
!guardrails hub install hub://guardrails/saliency_check --quiet
!guardrails hub install hub://guardrails/extracted_summary_sentences_match --quiet
!guardrails hub install hub://guardrails/similar_to_document --quiet
!guardrails hub install hub://guardrails/relevancy_evaluator --quiet

## ReadingLevel Examples (01–04)

In [ ]:
# Example 01: PhD-level prose blocked for grade-5 audience
from guardrails.hub import ReadingLevel
guard = Guard().use(ReadingLevel(reading_level=5, on_fail=OnFailAction.EXCEPTION))
complex_text = (
    'The epistemological implications of quantum superposition necessitate '
    'a paradigmatic reconfiguration of ontological presuppositions inherent '
    'within classical deterministic frameworks.'
)
try:
    guard.validate(complex_text)
except ValidationError:
    print('FAIL - text too complex for grade-5 reading level')

In [ ]:
# Example 02: Simple explanation at correct grade level passes
from guardrails.hub import ReadingLevel
guard = Guard().use(ReadingLevel(reading_level=5, on_fail=OnFailAction.EXCEPTION))
simple_text = 'The sun is a big star. It gives us light and heat. Plants need the sun to grow.'
outcome = guard.validate(simple_text)
print('PASS - appropriate reading level:', outcome.validation_passed)

In [ ]:
# Example 03: Children's app content — targeting grade 3
from guardrails.hub import ReadingLevel
guard = Guard().use(ReadingLevel(reading_level=3, on_fail=OnFailAction.NOOP))
kids_text = 'Dogs like to run and play. They wag their tails when they are happy.'
outcome = guard.validate(kids_text)
print('grade-3 kids text passed:', outcome.validation_passed)

In [ ]:
# Example 04: Technical documentation — college-level acceptable
from guardrails.hub import ReadingLevel
guard = Guard().use(ReadingLevel(reading_level=14, on_fail=OnFailAction.EXCEPTION))  # college level
tech_doc = (
    'The API rate limiter implements a token bucket algorithm to throttle requests '
    'based on client tier. Configure max_tokens and refill_rate parameters.'
)
outcome = guard.validate(tech_doc)
print('PASS - college-level tech doc:', outcome.validation_passed)

## ReadingTime Examples (05–07)

In [ ]:
# Example 05: Response too long for 1-minute reading time
from guardrails.hub import ReadingTime
guard = Guard().use(ReadingTime(reading_time=1, on_fail=OnFailAction.EXCEPTION))  # max 1 minute
long_text = ' '.join(['This is a very long document with many words.'] * 100)  # ~500+ words
try:
    guard.validate(long_text)
except ValidationError:
    print('FAIL - response exceeds 1-minute reading time')

In [ ]:
# Example 06: Short response within 2-minute reading window passes
from guardrails.hub import ReadingTime
guard = Guard().use(ReadingTime(reading_time=2, on_fail=OnFailAction.EXCEPTION))
short_text = 'Python is a versatile language used for web development, data science, and automation.'
outcome = guard.validate(short_text)
print('PASS - within reading time budget:', outcome.validation_passed)

In [ ]:
# Example 07: Custom WPM for technical audience (slower reading speed)
from guardrails.hub import ReadingTime
# Technical content read at ~150 WPM (slower than general 250 WPM)
guard = Guard().use(ReadingTime(reading_time=3, wpm=150, on_fail=OnFailAction.NOOP))
technical = ' '.join(['Configure the distributed ledger consensus mechanism.'] * 10)
outcome = guard.validate(technical)
print('custom WPM reading time result:', outcome.validation_passed)

## GibberishText Examples (08–11)

In [ ]:
# Example 08: Random character string flagged as gibberish
from guardrails.hub import GibberishText
guard = Guard().use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('asjdhaksjdhaksjdhaslkdhaslkdh sjdhasjkdhasjkdhas')
except ValidationError:
    print('FAIL - random characters detected as gibberish')

In [ ]:
# Example 09: Keyboard smash pattern detected
from guardrails.hub import GibberishText
guard = Guard().use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('qwerty asdfgh zxcvbn poiuyt lkjhg mnbvc')
except ValidationError:
    print('FAIL - keyboard smash detected')

In [ ]:
# Example 10: Valid Spanish sentence should not trigger GibberishText
from guardrails.hub import GibberishText
guard = Guard().use(GibberishText(threshold=0.5, on_fail=OnFailAction.NOOP))
spanish = 'El cielo es azul y el sol brilla con fuerza hoy.'
outcome = guard.validate(spanish)
print('Spanish text gibberish result (should pass):', outcome.validation_passed)

In [ ]:
# Example 11: LLM hallucinated nonsense (token artifacts)
from guardrails.hub import GibberishText
guard = Guard().use(GibberishText(threshold=0.4, on_fail=OnFailAction.EXCEPTION))
artifacts = 'The result is... florp zibble krand wumbo the final output norx.'
try:
    guard.validate(artifacts)
except ValidationError:
    print('FAIL - LLM token artifacts detected as gibberish')

## RedundantSentences Examples (12–15)

In [ ]:
# Example 12: Repetitive summary — same point stated three ways
from guardrails.hub import RedundantSentences
guard = Guard().use(RedundantSentences(threshold=0.7, on_fail=OnFailAction.EXCEPTION))
repetitive = (
    'Python is a great programming language. '
    'Python is an excellent programming language indeed. '
    'Python really is an amazing programming language.'
)
try:
    guard.validate(repetitive)
except ValidationError:
    print('FAIL - redundant sentences detected')

In [ ]:
# Example 13: Threshold tuning — 0.9 vs 0.7 on same text
from guardrails.hub import RedundantSentences
text = 'Water is essential for life. Hydration is critical for survival. Staying hydrated keeps you healthy.'
for threshold in [0.7, 0.85, 0.95]:
    guard = Guard().use(RedundantSentences(threshold=threshold, on_fail=OnFailAction.NOOP))
    outcome = guard.validate(text)
    print(f'  threshold={threshold}  passed={outcome.validation_passed}')

In [ ]:
# Example 14: Concise response with no redundancy passes
from guardrails.hub import RedundantSentences
guard = Guard().use(RedundantSentences(threshold=0.7, on_fail=OnFailAction.EXCEPTION))
concise = (
    'Machine learning enables computers to learn from data. '
    'Neural networks are a subset of ML inspired by the brain. '
    'Deep learning uses many-layered neural networks for complex tasks.'
)
outcome = guard.validate(concise)
print('PASS - no redundancy:', outcome.validation_passed)

In [ ]:
# Example 15: FAQ answer with common support response redundancy
from guardrails.hub import RedundantSentences
guard = Guard().use(RedundantSentences(threshold=0.75, on_fail=OnFailAction.NOOP))
faq_response = (
    'Thank you for reaching out to us. '
    'We appreciate you contacting our support team. '
    'Thanks for getting in touch with us today.'
)
outcome = guard.validate(faq_response)
print('FAQ redundancy detected (should fail):', not outcome.validation_passed)

## SaliencyCheck Examples (16–19)

In [ ]:
# Example 16: Summary contains irrelevant off-topic paragraph
from guardrails.hub import SaliencyCheck
guard = Guard().use(SaliencyCheck(threshold=0.25, on_fail=OnFailAction.EXCEPTION))
document = 'Black holes are regions of spacetime where gravity is so strong that nothing can escape.'
irrelevant_summary = 'Black holes have immense gravity. By the way, pizza is a popular food worldwide.'
try:
    guard.validate(irrelevant_summary, metadata={'document': document})
except ValidationError:
    print('FAIL - off-topic sentence detected in summary')

In [ ]:
# Example 17: All sentences relevant to source document
from guardrails.hub import SaliencyCheck
guard = Guard().use(SaliencyCheck(threshold=0.25, on_fail=OnFailAction.EXCEPTION))
document = 'Black holes are regions of spacetime where gravity is so strong that nothing can escape.'
good_summary = 'Black holes are characterized by extremely strong gravitational forces from which no matter can escape.'
outcome = guard.validate(good_summary, metadata={'document': document})
print('PASS - all sentences salient:', outcome.validation_passed)

In [ ]:
# Example 18: Saliency check with metadata document parameter
from guardrails.hub import SaliencyCheck
guard = Guard().use(SaliencyCheck(threshold=0.2, on_fail=OnFailAction.NOOP))
product_doc = 'The XR-5000 camera features a 50MP sensor, 8K video, and 10x optical zoom.'
review = 'The XR-5000 offers exceptional image quality with its 50MP sensor and 8K video capability.'
outcome = guard.validate(review, metadata={'document': product_doc})
print('product review saliency:', outcome.validation_passed)

In [ ]:
# Example 19: News article saliency check for journalism
from guardrails.hub import SaliencyCheck
guard = Guard().use(SaliencyCheck(threshold=0.2, on_fail=OnFailAction.NOOP))
article = 'A new climate study shows global temperatures rose by 1.5C since pre-industrial times.'
on_topic = 'Global temperatures have increased 1.5 degrees Celsius compared to pre-industrial levels.'
outcome = guard.validate(on_topic, metadata={'document': article})
print('news saliency check:', outcome.validation_passed)

## ExtractedSummarySentencesMatch Examples (20–22)

In [ ]:
# Example 20: Good extractive summary — all sentences from source
from guardrails.hub import ExtractedSummarySentencesMatch
source = (
    'The Amazon rainforest covers over 5.5 million square kilometers. '
    'It is home to an estimated 10% of all species on Earth. '
    'Deforestation threatens its biodiversity every year.'
)
guard = Guard().use(ExtractedSummarySentencesMatch(threshold=0.85, on_fail=OnFailAction.EXCEPTION))
extractive = 'The Amazon rainforest covers over 5.5 million square kilometers. Deforestation threatens its biodiversity every year.'
outcome = guard.validate(extractive, metadata={'filepaths': [source]})
print('PASS - extractive summary valid:', outcome.validation_passed)

In [ ]:
# Example 21: Added invented sentence mixed into extractive summary
from guardrails.hub import ExtractedSummarySentencesMatch
source = 'The Amazon rainforest covers over 5.5 million square kilometers and hosts 10% of Earth species.'
guard = Guard().use(ExtractedSummarySentencesMatch(threshold=0.85, on_fail=OnFailAction.EXCEPTION))
with_invented = source + ' Scientists have recently discovered time travel in the Amazon.'
try:
    guard.validate(with_invented, metadata={'filepaths': [source]})
except ValidationError:
    print('FAIL - invented sentence detected in extractive summary')

In [ ]:
# Example 22: Slightly rephrased sentence — paraphrase detection
from guardrails.hub import ExtractedSummarySentencesMatch
source = 'Python was created by Guido van Rossum and first released in 1991.'
guard = Guard().use(ExtractedSummarySentencesMatch(threshold=0.8, on_fail=OnFailAction.NOOP))
paraphrased = 'Guido van Rossum developed Python, which made its debut in 1991.'
outcome = guard.validate(paraphrased, metadata={'filepaths': [source]})
print('paraphrase similarity result:', outcome.validation_passed)

## SimilarToDocument Examples (23–25)

In [ ]:
# Example 23: High similarity — output very close to reference
from guardrails.hub import SimilarToDocument
reference = 'Our product offers enterprise-grade security with end-to-end encryption and SOC2 compliance.'
guard = Guard().use(SimilarToDocument(document=reference, threshold=0.6, on_fail=OnFailAction.EXCEPTION))
similar = 'The product provides enterprise security features including encryption and compliance certifications.'
outcome = guard.validate(similar)
print('PASS - high similarity:', outcome.validation_passed)

In [ ]:
# Example 24: Low similarity — output diverges significantly from reference
from guardrails.hub import SimilarToDocument
reference = 'Our product offers enterprise-grade security with end-to-end encryption.'
guard = Guard().use(SimilarToDocument(document=reference, threshold=0.8, on_fail=OnFailAction.EXCEPTION))
unrelated = 'I enjoy hiking in the mountains during summer vacation with my friends.'
try:
    guard.validate(unrelated)
except ValidationError:
    print('FAIL - output too dissimilar from reference document')

In [ ]:
# Example 25: Brand voice enforcement — company tone guide as reference
from guardrails.hub import SimilarToDocument
brand_voice = (
    'At Acme Corp, we believe in simple, friendly communication. '
    'We speak clearly, avoid jargon, and put customers first.'
)
guard = Guard().use(SimilarToDocument(document=brand_voice, threshold=0.5, on_fail=OnFailAction.NOOP))
response = 'Hi there! We are here to help you get the most out of your experience with us.'
outcome = guard.validate(response)
print('brand voice similarity:', outcome.validation_passed)

## RelevancyEvaluator Examples (26–29)

In [ ]:
# Example 26: On-topic LLM response — answer directly addresses the question
from guardrails.hub import RelevancyEvaluator
guard = Guard().use(
    RelevancyEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
question = 'What is the capital of Japan?'
answer = 'The capital of Japan is Tokyo, a major global metropolis.'
outcome = guard.validate(answer, metadata={'original_prompt': question})
print('PASS - relevant answer:', outcome.validation_passed)

In [ ]:
# Example 27: Off-topic tangent — response wanders from user question
from guardrails.hub import RelevancyEvaluator
guard = Guard().use(
    RelevancyEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
question = 'How do I fix a null pointer exception in Java?'
irrelevant_answer = 'Java is an island in Indonesia known for its coffee and temples. Many tourists visit annually.'
try:
    guard.validate(irrelevant_answer, metadata={'original_prompt': question})
except ValidationError:
    print('FAIL - response irrelevant to the question')

In [ ]:
# Example 28: metadata original_prompt parameter pattern
from guardrails.hub import RelevancyEvaluator
guard = Guard().use(
    RelevancyEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
outcome = guard(
    oai.chat.completions.create,
    prompt='Explain what a REST API is.',
    model=MODEL,
    metadata={'original_prompt': 'Explain what a REST API is.'}
)
print('LLM response relevant:', outcome.validation_passed)

In [ ]:
# Example 29: Customer support deflection — bot answers different question
from guardrails.hub import RelevancyEvaluator
guard = Guard().use(
    RelevancyEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
user_q = 'My order #12345 has not arrived. When will it be delivered?'
deflection = 'Our company was founded in 2010 and serves customers worldwide. We have great products!'
try:
    guard.validate(deflection, metadata={'original_prompt': user_q})
except ValidationError:
    print('FAIL - support deflection detected (response ignores order question)')

## Combined Quality Pipelines (30–32)

In [ ]:
# Example 30: ReadingLevel + ReadingTime combo for content platform
from guardrails.hub import ReadingLevel, ReadingTime
guard = (
    Guard()
    .use(ReadingLevel(reading_level=8, on_fail=OnFailAction.NOOP))
    .use(ReadingTime(reading_time=3, on_fail=OnFailAction.NOOP))
)
content = (
    'Social media has changed how people communicate. '
    'Users share photos, videos, and text with their followers. '
    'Businesses use these platforms to reach customers directly.'
)
outcome = guard.validate(content)
print('content platform quality check:', outcome.validation_passed)

In [ ]:
# Example 31: GibberishText + RelevancyEvaluator — dual quality guard for LLM output
from guardrails.hub import GibberishText, RelevancyEvaluator
guard = (
    Guard()
    .use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
    .use(RelevancyEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP))
)
outcome = guard(
    oai.chat.completions.create,
    prompt='What are the benefits of exercise?',
    model=MODEL,
    metadata={'original_prompt': 'What are the benefits of exercise?'}
)
print('dual quality check passed:', outcome.validation_passed)

In [ ]:
# Example 32: Full text quality pipeline — ReadingLevel + GibberishText + RedundantSentences + ReadingTime
from guardrails.hub import ReadingLevel, GibberishText, RedundantSentences, ReadingTime
quality_guard = (
    Guard()
    .use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
    .use(ReadingLevel(reading_level=10, on_fail=OnFailAction.NOOP))
    .use(RedundantSentences(threshold=0.8, on_fail=OnFailAction.NOOP))
    .use(ReadingTime(reading_time=5, on_fail=OnFailAction.NOOP))
)
outcome = quality_guard(
    oai.chat.completions.create,
    prompt='Write a short paragraph about renewable energy.',
    model=MODEL
)
print('4-validator quality pipeline passed:', outcome.validation_passed)
print('output:', outcome.validated_output[:150] if outcome.validated_output else 'None')